# 00 · Setup check

Run all cells of this notebook before the workshop. It checks that the Python packages are installed and that a tiny spiking network runs on the Brian2 simulator. It takes about 10 seconds and needs no SpiNNaker2 hardware.

If a cell fails, see the install guide in the [README](../README.md) or bring the error message to the workshop.

## 1. Packages

In [ ]:
import importlib
import importlib.metadata

for name in ["spinnaker2", "brian2", "torch", "snntorch", "nir", "nirtorch", "numpy", "matplotlib"]:
    try:
        importlib.import_module(name)
        print(f"{name:12s} {importlib.metadata.version(name)}")
    except ImportError:
        print(f"{name:12s} MISSING")

## 2. A tiny network on Brian2

One input neuron sends three spikes to one LIF neuron. The LIF neuron should answer each input spike with one spike of its own, one step later: the time the neuron needs to process the spike.

In [ ]:
from spinnaker2 import brian2_sim, snn

# input neuron that spikes at time steps 1, 5 and 9 (one step = 1 ms)
input_times = [1, 5, 9]
stimulus = snn.Population(size=1, neuron_model="spike_list", params={0: input_times}, name="input")

# LIF neuron: spikes when its membrane potential exceeds the threshold, then resets to 0
neuron = snn.Population(
    size=1,
    neuron_model="lif",
    params={"threshold": 1.0, "alpha_decay": 0.9, "reset": "reset_to_v_reset", "v_reset": 0.0},
    name="lif",
    record=["spikes"],  # keep the spike times
)

# connection [pre, post, weight, delay]: weight 2 > threshold 1, so every input spike triggers an output spike
connection = snn.Projection(pre=stimulus, post=neuron, connections=[[0, 0, 2, 0]])

network = snn.Network("setup_check")
network.add(stimulus, neuron, connection)
brian2_sim.Brian2Backend().run(network, 20)  # simulate 20 time steps

spikes = neuron.get_spikes()[0]
print("Output spikes:", spikes)
print("Brian2 works." if list(spikes) == [2, 6, 10] else "Unexpected spikes: please bring this notebook to the workshop.")

## 3. What happened

Each output spike (black) follows an input spike (grey) one step later. If the plot looks like this, the setup works.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 1.5))
plt.eventplot([spikes, input_times], colors=["black", "grey"])
plt.yticks([0, 1], ["output", "input"])
plt.xticks(range(0, 21, 2))
plt.xlabel("time (steps of 1 ms)")
plt.show()